# Goblin Colab Worker

Disposable GPU inference worker for Goblin. Serves an OpenAI-compatible API via Cloudflare Tunnel.

**Before running:** Make sure Runtime → Change runtime type → GPU is set to **T4** (or better).

In [ ]:
# Step 1 — Verify GPU
!nvidia-smi
# If you see K80, disconnect and reconnect until you get a T4.

In [ ]:
# Step 2 — Install dependencies (CUDA build of llama-cpp-python)
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install llama-cpp-python huggingface_hub fastapi uvicorn nest_asyncio -q

In [ ]:
# Step 3 — Set credentials
import os

# Paste the value of COLAB_WORKER_API_KEY from your backend .env here
os.environ["COLAB_WORKER_API_KEY"] = "iU4FHYpikd8tq--pGNyonE8yLMx5xAz-LgXyrCcI9to"

API_KEY = os.environ["COLAB_WORKER_API_KEY"]

In [ ]:
# Step 4 — Download GGUF model from Hugging Face
# ~15-30 seconds. Cached in /tmp/hf_cache for the runtime lifetime.
from huggingface_hub import hf_hub_download

# Default: gemma-3-12b (fast, general chat/research)
model_path = hf_hub_download(
    repo_id="bartowski/gemma-3-12b-it-GGUF",
    filename="gemma-3-12b-it-Q4_K_M.gguf",
    cache_dir="/tmp/hf_cache",
)

# Alternate: qwen3-14b (coding/planning/agent tasks)
# model_path = hf_hub_download(
#     repo_id="bartowski/Qwen3-14B-GGUF",
#     filename="Qwen3-14B-Q4_K_M.gguf",
#     cache_dir="/tmp/hf_cache",
# )

print("Model ready:", model_path)

In [ ]:
# Step 5 — Build FastAPI app with concurrency guard
import asyncio
from fastapi import FastAPI, Header, HTTPException
from llama_cpp import Llama

app = FastAPI()
sem = asyncio.Semaphore(2)  # max 2 concurrent generations

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,   # offload all layers to GPU
    n_ctx=2048,
    n_batch=512,
    use_mlock=True,
    use_mmap=True,
)

def _auth(authorization: str | None) -> None:
    if authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="unauthorized")

@app.get("/health")
def health(authorization: str | None = Header(default=None)):
    _auth(authorization)
    return {"healthy": True}

@app.get("/v1/models")
def models(authorization: str | None = Header(default=None)):
    _auth(authorization)
    return {"object": "list", "data": [{"id": "gemma-3-12b", "object": "model"}]}

@app.post("/v1/chat/completions")
async def chat(req: dict, authorization: str | None = Header(default=None)):
    _auth(authorization)
    messages = req.get("messages") or []
    prompt = "\n".join(m.get("content", "") for m in messages if isinstance(m, dict))
    async with sem:
        output = llm(prompt, max_tokens=req.get("max_tokens", 512))
    text = output["choices"][0]["text"]
    return {
        "id": "chatcmpl-colab",
        "object": "chat.completion",
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
    }

print("App defined.")

In [ ]:
# Step 6 — Start server in background
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

t = threading.Thread(target=_run, daemon=True)
t.start()
print("Server started on port 8000")

In [ ]:
# Step 7 — Start Cloudflare Tunnel and print public URL
import subprocess, re, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

cf_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stderr=subprocess.PIPE,
    text=True,
)

tunnel_url = None
for line in cf_proc.stderr:
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break

print()
print("=" * 60)
print("COLAB_WORKER_ENDPOINT =", tunnel_url)
print("=" * 60)
print()
print("Set this as COLAB_WORKER_ENDPOINT in your backend .env, then restart the backend.")

In [ ]:
# Step 8 — Register tunnel URL with the running backend (no redeploy needed)
import requests as _req

# Switch to "http://localhost:8001" when testing locally
BACKEND_URL = "https://goblinos-assistant-backend-v2.vercel.app"

_rh = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

resp = _req.post(
    f"{BACKEND_URL}/ops/colab-worker/register",
    headers=_rh,
    json={"endpoint": tunnel_url},
    timeout=20,
)

if resp.status_code == 200:
    data = resp.json()
    print("Registered:", data["endpoint"])
    print("Health probe:", data.get("health"))
    print("Env written:", data.get("env_written"))
else:
    print("Registration failed:", resp.status_code, resp.text)
    print()
    print("Falling back: updating Render env var for persistence across backend restarts...")

    RENDER_API_KEY = "rnd_qGJtRCADU9mKwQXie9slewaPMtoa"
    RENDER_SERVICE_ID = "srv-d4klibc9c44c73f1jh00"
    _render_h = {"Authorization": f"Bearer {RENDER_API_KEY}", "Accept": "application/json"}
    _base = f"https://api.render.com/v1/services/{RENDER_SERVICE_ID}"

    _evs = _req.get(f"{_base}/env-vars", headers=_render_h).json()
    _new = [{"key": e["envVar"]["key"], "value": e["envVar"]["value"]} for e in _evs]
    for ev in _new:
        if ev["key"] == "COLAB_WORKER_ENDPOINT":
            ev["value"] = tunnel_url
            break
    else:
        _new.append({"key": "COLAB_WORKER_ENDPOINT", "value": tunnel_url})
    r = _req.put(f"{_base}/env-vars", headers={**_render_h, "Content-Type": "application/json"}, json=_new)
    print("Render env var update:", r.status_code)
    r = _req.post(f"{_base}/deploys", headers={**_render_h, "Content-Type": "application/json"}, json={"clearCache": "do_not_clear"})
    print("Redeploy triggered:", r.status_code, "— backend ready in ~1 min")

In [ ]:
# Step 9 — Self-test the worker directly
import requests

headers = {"Authorization": f"Bearer {API_KEY}"}

r = requests.get(f"{tunnel_url}/health", headers=headers)
print("Health:", r.json())

r = requests.post(
    f"{tunnel_url}/v1/chat/completions",
    headers={**headers, "Content-Type": "application/json"},
    json={"model": "gemma-3-12b", "messages": [{"role": "user", "content": "Say hello in one sentence."}]},
)
print("Response:", r.json()["choices"][0]["message"]["content"])

## Keep-alive

Paste this into the browser DevTools console (F12) in the Colab tab:

```javascript
function clickConnect() {
    document.querySelector('#top-toolbar > colab-connect-button')
        .shadowRoot.querySelector('#connect').click()
}
setInterval(clickConnect, 60000)
```

For a more robust setup, configure [UptimeRobot](https://uptimerobot.com) to ping `<TUNNEL_URL>/health` every 5 minutes.